# 05 — Bagging / 装袋方法

**Chapter 14 — File 5 of 7 / 第14章 — 第5个文件（共7个）**

---

## Summary / 总结

This script demonstrates **Import Bagging Regressor and compare how performance is affected by Bagging**.

本脚本演示 **Import Bagging Regressor and compare how performance is affected by Bagging**。

---
## Step 1 — Import Bagging Regressor and compare how performance is affected by Bagging
(i.e. increasing number of trees)

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import BaggingRegressor

---
## Step 2 — Load the dataset

In [ ]:
Ames = pd.read_csv("Ames.csv")

---
## Step 3 — Convert the below numeric features to categorical features

In [ ]:
Ames["MSSubClass"] = Ames["MSSubClass"].astype("object")
Ames["YrSold"] = Ames["YrSold"].astype("object")
Ames["MoSold"] = Ames["MoSold"].astype("object")

---
## Step 4 — Identify numeric and categorical features from the dataset based on the data type

In [ ]:
numeric_features = Ames.select_dtypes(include=["int64", "float64"]) \
                       .drop(columns=["PID", "SalePrice"]).columns
categorical_features = Ames.select_dtypes(include=["object"]).columns \
                           .difference(["Electrical"])

---
## Step 5 — Manually specify the categories for ordinal encoding according to the data dictionary

In [ ]:
ordinal_order = {

---
## Step 6 — Electrical system

In [ ]:
"Electrical": ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],

---
## Step 7 — General shape of property

In [ ]:
"LotShape": ["IR3", "IR2", "IR1", "Reg"],

---
## Step 8 — Type of utilities available

In [ ]:
"Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],

---
## Step 9 — Slope of property

In [ ]:
"LandSlope": ["Sev", "Mod", "Gtl"],

---
## Step 10 — Evaluates the quality of the material on the exterior

In [ ]:
"ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 11 — Evaluates the present condition of the material on the exterior

In [ ]:
"ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 12 — Height of the basement

In [ ]:
"BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 13 — General condition of the basement

In [ ]:
"BsmtCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 14 — Walkout or garden level basement walls

In [ ]:
"BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],

---
## Step 15 — Quality of basement finished area

In [ ]:
"BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

---
## Step 16 — Quality of second basement finished area

In [ ]:
"BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],

---
## Step 17 — Heating quality and condition

In [ ]:
"HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 18 — Kitchen quality

In [ ]:
"KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 19 — Home functionality

In [ ]:
"Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],

---
## Step 20 — Fireplace quality

In [ ]:
"FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 21 — Interior finish of the garage

In [ ]:
"GarageFinish": ["None", "Unf", "RFn", "Fin"],

---
## Step 22 — Garage quality

In [ ]:
"GarageQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 23 — Garage condition

In [ ]:
"GarageCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],

---
## Step 24 — Paved driveway

In [ ]:
"PavedDrive": ["N", "P", "Y"],

---
## Step 25 — Pool quality

In [ ]:
"PoolQC": ["None", "Fa", "TA", "Gd", "Ex"],

---
## Step 26 — Fence quality

In [ ]:
"Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"]
}

---
## Step 27 — Extract list of ALL ordinal features from dictionary

In [ ]:
ordinal_features = list(ordinal_order.keys())

---
## Step 28 — List of ordinal features except Electrical

In [ ]:
ordinal_except_electrical = [feat for feat in ordinal_features if feat != "Electrical"]

---
## Step 29 — Helper function to fill "None" for missing categorical data

In [ ]:
def fill_none(X):
    return X.infer_objects(copy=False).fillna("None")

---
## Step 30 — Pipeline for "Electrical": Fill missing value with mode then apply ordinal encoding

In [ ]:
electrical_transformer = Pipeline(steps=[
    ("impute_electrical", SimpleImputer(strategy="most_frequent")),
    ("ordinal_electrical", OrdinalEncoder(categories=[ordinal_order["Electrical"]]))
])

---
## Step 31 — Pipeline for numeric features: Impute missing values using mean

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("impute_mean", SimpleImputer(strategy="mean"))
])

---
## Step 32 — Pipeline for ordinal features: Fill missing values with "None" then apply
ordinal encoding

In [ ]:
ordinal_transformer = Pipeline(steps=[
    ("fill_none", FunctionTransformer(fill_none, validate=False)),
    ("ordinal", OrdinalEncoder(categories=[ordinal_order[feat]
                                           for feat in ordinal_features
                                           if feat in ordinal_except_electrical]))
])

---
## Step 33 — Pipeline for nominal categorical features: Fill missing values with "None" then apply
one-hot encoding

In [ ]:
nominal_features = [feat for feat in categorical_features if feat not in ordinal_features]
categorical_transformer = Pipeline(steps=[
    ("fill_none", FunctionTransformer(fill_none, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

---
## Step 34 — Combined preprocessor for numeric, ordinal, nominal, and specific electrical data

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("electrical", electrical_transformer, ["Electrical"]),
        ("num", numeric_transformer, numeric_features),
        ("ordinal", ordinal_transformer, ordinal_except_electrical),
        ("nominal", categorical_transformer, nominal_features)
])

models = {
    "Decision Tree (1 Tree)":
        DecisionTreeRegressor(random_state=42),
    "Bagging Regressor (10 Trees)":
        BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                         n_estimators=10, random_state=42)
}

results = {}
for name, model in models.items():

---
## Step 35 — Define the full model pipeline for each model

In [ ]:
model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])

---
## Step 36 — Perform cross-validation

In [ ]:
scores = cross_val_score(model_pipeline,
                             Ames.drop(columns="SalePrice"), Ames["SalePrice"])

---
## Step 37 — Store and print the mean of the scores

In [ ]:
results[name] = round(scores.mean(), 4)

---
## Step 38 — Output the cross-validation scores

In [ ]:
print("Cross-validation scores:", results)

---
## Learning Notes / 学习笔记

- **概念**: Import Bagging Regressor and compare how performance is affected by Bagging 是机器学习中的常用技术。  
  *Import Bagging Regressor and compare how performance is affected by Bagging is a common technique in machine learning.*

- **ML 应用**: 本示例展示了如何在实践中应用该技术。  
  *This example shows how to apply the technique in practice.*

---
## Complete Code / 完整代码一览

Below is the full code for quick reference. / 以下是完整代码，供快速参考。

In [ ]:
# ===============================
# Bagging / 装袋方法
# Complete Code / 完整代码
# ===============================

# Import Bagging Regressor and compare how performance is affected by Bagging
# (i.e. increasing number of trees)

import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import BaggingRegressor

# Load the dataset
Ames = pd.read_csv("Ames.csv")

# Convert the below numeric features to categorical features
Ames["MSSubClass"] = Ames["MSSubClass"].astype("object")
Ames["YrSold"] = Ames["YrSold"].astype("object")
Ames["MoSold"] = Ames["MoSold"].astype("object")

# Identify numeric and categorical features from the dataset based on the data type
numeric_features = Ames.select_dtypes(include=["int64", "float64"]) \
                       .drop(columns=["PID", "SalePrice"]).columns
categorical_features = Ames.select_dtypes(include=["object"]).columns \
                           .difference(["Electrical"])

# Manually specify the categories for ordinal encoding according to the data dictionary
ordinal_order = {
    # Electrical system
    "Electrical": ["Mix", "FuseP", "FuseF", "FuseA", "SBrkr"],
    # General shape of property
    "LotShape": ["IR3", "IR2", "IR1", "Reg"],
    # Type of utilities available
    "Utilities": ["ELO", "NoSeWa", "NoSewr", "AllPub"],
    # Slope of property
    "LandSlope": ["Sev", "Mod", "Gtl"],
    # Evaluates the quality of the material on the exterior
    "ExterQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Evaluates the present condition of the material on the exterior
    "ExterCond": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Height of the basement
    "BsmtQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # General condition of the basement
    "BsmtCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Walkout or garden level basement walls
    "BsmtExposure": ["None", "No", "Mn", "Av", "Gd"],
    # Quality of basement finished area
    "BsmtFinType1": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    # Quality of second basement finished area
    "BsmtFinType2": ["None", "Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
    # Heating quality and condition
    "HeatingQC": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Kitchen quality
    "KitchenQual": ["Po", "Fa", "TA", "Gd", "Ex"],
    # Home functionality
    "Functional": ["Sal", "Sev", "Maj2", "Maj1", "Mod", "Min2", "Min1", "Typ"],
    # Fireplace quality
    "FireplaceQu": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Interior finish of the garage
    "GarageFinish": ["None", "Unf", "RFn", "Fin"],
    # Garage quality
    "GarageQual": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Garage condition
    "GarageCond": ["None", "Po", "Fa", "TA", "Gd", "Ex"],
    # Paved driveway
    "PavedDrive": ["N", "P", "Y"],
    # Pool quality
    "PoolQC": ["None", "Fa", "TA", "Gd", "Ex"],
    # Fence quality
    "Fence": ["None", "MnWw", "GdWo", "MnPrv", "GdPrv"]
}

# Extract list of ALL ordinal features from dictionary
ordinal_features = list(ordinal_order.keys())

# List of ordinal features except Electrical
ordinal_except_electrical = [feat for feat in ordinal_features if feat != "Electrical"]

# Helper function to fill "None" for missing categorical data
def fill_none(X):
    return X.infer_objects(copy=False).fillna("None")

# Pipeline for "Electrical": Fill missing value with mode then apply ordinal encoding
electrical_transformer = Pipeline(steps=[
    ("impute_electrical", SimpleImputer(strategy="most_frequent")),
    ("ordinal_electrical", OrdinalEncoder(categories=[ordinal_order["Electrical"]]))
])

# Pipeline for numeric features: Impute missing values using mean
numeric_transformer = Pipeline(steps=[
    ("impute_mean", SimpleImputer(strategy="mean"))
])

# Pipeline for ordinal features: Fill missing values with "None" then apply
# ordinal encoding
ordinal_transformer = Pipeline(steps=[
    ("fill_none", FunctionTransformer(fill_none, validate=False)),
    ("ordinal", OrdinalEncoder(categories=[ordinal_order[feat]
                                           for feat in ordinal_features
                                           if feat in ordinal_except_electrical]))
])

# Pipeline for nominal categorical features: Fill missing values with "None" then apply
# one-hot encoding
nominal_features = [feat for feat in categorical_features if feat not in ordinal_features]
categorical_transformer = Pipeline(steps=[
    ("fill_none", FunctionTransformer(fill_none, validate=False)),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combined preprocessor for numeric, ordinal, nominal, and specific electrical data
preprocessor = ColumnTransformer(
    transformers=[
        ("electrical", electrical_transformer, ["Electrical"]),
        ("num", numeric_transformer, numeric_features),
        ("ordinal", ordinal_transformer, ordinal_except_electrical),
        ("nominal", categorical_transformer, nominal_features)
])

models = {
    "Decision Tree (1 Tree)":
        DecisionTreeRegressor(random_state=42),
    "Bagging Regressor (10 Trees)":
        BaggingRegressor(estimator=DecisionTreeRegressor(random_state=42),
                         n_estimators=10, random_state=42)
}

results = {}
for name, model in models.items():
    # Define the full model pipeline for each model
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", model)
    ])

    # Perform cross-validation
    scores = cross_val_score(model_pipeline,
                             Ames.drop(columns="SalePrice"), Ames["SalePrice"])

    # Store and print the mean of the scores
    results[name] = round(scores.mean(), 4)

# Output the cross-validation scores
print("Cross-validation scores:", results)

---

➡️ **Next / 下一步**: File 6 of 7